Lip product analysis comparing original launch and revival shades.
*Co-authored with CoCo*

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

for weight_file in ['ClashGrotesk-Extralight.ttf', 'ClashGrotesk-Light.ttf',
                    'ClashGrotesk-Regular.ttf', 'ClashGrotesk-Medium.ttf',
                    'ClashGrotesk-Semibold.ttf', 'ClashGrotesk-Bold.ttf']:
    fm.fontManager.addfont(f'fonts/Clash_Grotesk/{weight_file}')

plt.rcParams['font.family'] = 'Clash Grotesk'
plt.rcParams['font.weight'] = 'light'

In [ ]:
%%sql -r dataframe_1
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r lip_products_data
SELECT
  era,
  launch_year,
  product_name,
  product_type,
  shade_code,
  shade_name,
  shade_description,
  finish,
  finish_group,
  color_family,
  shade_depth_group,
  undertone_inferred_cmyk,
  undertone_confidence,
  c_pct,
  m_pct,
  y_pct,
  k_pct,
  color_value_source
FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.SHADES
WHERE product_type IN ('Lipstick', 'Lip Gloss', 'Lip Balm')
  AND c_pct IS NOT NULL
  AND m_pct IS NOT NULL
  AND y_pct IS NOT NULL
  AND k_pct IS NOT NULL
ORDER BY era, product_type, product_name, shade_name;

In [ ]:
import pandas as pd
from matplotlib.lines import Line2D

def cmyk_to_rgb(c, m, y, k):
    c, m, y, k = c / 100, m / 100, y / 100, k / 100
    r = (1 - c) * (1 - k)
    g = (1 - m) * (1 - k)
    b = (1 - y) * (1 - k)
    return (r, g, b)

df = lip_products_data.to_pandas() if hasattr(lip_products_data, 'to_pandas') else lip_products_data
df = df.copy()
df.columns = [col.lower() for col in df.columns]

df['rgb'] = df.apply(lambda row: cmyk_to_rgb(row['c_pct'], row['m_pct'], row['y_pct'], row['k_pct']), axis=1)
df['darkness'] = df['k_pct']
df['hue_direction'] = df['m_pct'] - df['y_pct']

shape_map = {
    'Lipstick': 'o',
    'Lip Gloss': '^',
}

era_order = ['original_launch', 'revival']
era_labels = {'original_launch': '2013 Collection', 'revival': '2026 Revival'}
era_annotations = {
    'original_launch': 'Glosses and one lipstick span pink-to-warm across light-to-dark',
    'revival': 'Single bold lipstick in a deeper, pinker formulation',
}

def get_extreme_labels(era_df, n=4):
    extremes = set()
    if len(era_df) <= n * 2:
        return set(era_df['shade_name'])
    extremes.update(era_df.nsmallest(n, 'darkness')['shade_name'])
    extremes.update(era_df.nlargest(n, 'darkness')['shade_name'])
    extremes.update(era_df.nsmallest(2, 'hue_direction')['shade_name'])
    extremes.update(era_df.nlargest(2, 'hue_direction')['shade_name'])
    return extremes

def plot_lips(layout='horizontal'):
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
        order = era_order
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 14), sharex=True)
        order = era_order

    for i, (axis, era) in enumerate(zip(axes.flat, order)):
        era_df = df[df['era'] == era].copy()
        axis.axhline(0, color='#d9d9d9', linewidth=1, zorder=0)

        for product_type, type_df in era_df.groupby('product_type'):
            colors = list(type_df['rgb'])
            marker = shape_map.get(product_type, 'D')
            axis.scatter(
                type_df['darkness'],
                type_df['hue_direction'],
                s=120,
                alpha=0.85,
                c=colors,
                edgecolors='black',
                linewidths=0.5,
                marker=marker,
                label=product_type,
            )

        extremes = get_extreme_labels(era_df)
        for _, row in era_df.iterrows():
            if row['shade_name'] in extremes:
                axis.annotate(
                    row['shade_name'],
                    (row['darkness'], row['hue_direction']),
                    textcoords='offset points',
                    xytext=(5, 4),
                    fontsize=9,
                    fontweight='normal',
                    alpha=0.85,
                )

        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, era_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_xlabel('K% (darkness / depth)', fontsize=11, fontweight='normal')
        axis.grid(alpha=0.2)
        axis.tick_params(labelsize=9)

        if layout == 'vertical' or i == 0:
            axis.set_ylabel('M% - Y% (positive = pink, negative = warm)',
                            fontsize=11, fontweight='normal')

    legend_handles = [
        Line2D([0], [0], marker=m, color='black', markerfacecolor='white',
               linestyle='', markersize=9, label=pt)
        for pt, m in shape_map.items()
        if pt in df['product_type'].unique()
    ]
    if layout == 'horizontal':
        axes.flat[0].legend(handles=legend_handles, title='Product type',
                            loc='lower left', bbox_to_anchor=(0, 1.15),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        fig.suptitle('LIP PRODUCTS: FROM GLOSS-HEAVY TO A SINGLE BOLD LIPSTICK',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        axes.flat[0].legend(handles=legend_handles, title='Product type',
                            loc='lower left', bbox_to_anchor=(0, 1.08),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('LIP PRODUCTS: FROM GLOSS-HEAVY TO A SINGLE BOLD LIPSTICK',
                     fontsize=18, fontweight='medium', y=1.05)
    plt.show()

plot_lips('horizontal')
plot_lips('vertical')